# 13 · MLflow experiment tracking and model packaging

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** databricks. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Log parameters, metrics, a signature and an input example; reload and verify predictions.


## Step 1 — Prepare an experiment

The workspace experiment path belongs to the current user. The notebook logs one small sklearn model.


In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
X, y = make_classification(n_samples=1000, n_features=10, n_informative=5,
                           weights=[0.75, 0.25], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
user=spark.sql('SELECT current_user()').first()[0]
mlflow.set_tracking_uri('databricks')
mlflow.set_experiment(f'/Users/{user}/ai_engineering_hands_on')


**Check your result:** MLflow selects or creates your course experiment.


## Step 2 — Train and track

A signature documents model inputs/outputs. NumPy input examples avoid pandas dependencies in your code.


In [ ]:
model=make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=500))
model.fit(X_train,y_train)
prob=model.predict_proba(X_test)[:,1]
with mlflow.start_run(run_name='logistic_baseline') as run:
    mlflow.log_params({'C':1,'seed':42,'train_rows':len(X_train)})
    mlflow.log_metric('test_roc_auc',roc_auc_score(y_test,prob))
    info=mlflow.sklearn.log_model(model,artifact_path='model',
        signature=infer_signature(X_train,model.predict(X_train)),input_example=X_train[:3])
    model_uri=info.model_uri
    run_id=run.info.run_id
print('Run:',run_id,'model:',model_uri)


**Check your result:** The experiment contains parameters, a test metric, and a signed model artifact.


## Step 3 — Load the exact model artifact

Artifact reload is an essential packaging check, separate from model quality.


In [ ]:
loaded=mlflow.sklearn.load_model(model_uri)
assert np.array_equal(model.predict(X_test[:10]),loaded.predict(X_test[:10]))
print('Artifact round-trip passed')


**Check your result:** Loaded and original predictions agree.


## Practice

Log C=0.1 and C=10 in separate runs using a validation score. Compare them in the MLflow UI.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Use validation to select C and reserve the test metric for the final candidate. Record dataset version and feature schema as tags.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
